# Tema de Programare: Tratarea Datelor Dezechilibrate

Bine ai venit la tema despre tratarea datelor dezechilibrate.

Dezechilibrul de clasă este una dintre cele mai comune probleme cu care te vei confrunta în machine learning-ul din lumea reală. Detectarea fraudei, diagnosticul medical, detectarea defectelor — în toate aceste cazuri, ceea ce îți pasă cel mai mult să prezici este elementul rar. Un model care îl ignoră poate arăta excelent după acuratețe și totuși să fie complet inutil.

În această temă vei parcurge întregul pipeline pentru gestionarea datelor dezechilibrate: diagnosticarea problemei, corectarea ei și evaluarea corectă a modelului. Vei folosi tehnici reale de resamplare — ponderi de clasă, SMOTE, Tomek Links și NearMiss — și vei învăța care metrici de evaluare spun cu adevărat povestea corectă.

Tema se încheie cu o implementare de la zero a algoritmului SMOTE, astfel încât să înțelegi exact ce face biblioteca în culise.

**Ce vei face în această temă**

* Diagnostichezi dezechilibrul de clasă și cuantifici cât de sever este.
* Aplici ponderi de clasă pentru ca clasificatorii să acorde atenție clasei minoritare.
* Folosești SMOTE pentru a genera eșantioane sintetice minoritare și îi ajustezi parametrii.
* Aplici strategii de undersampling (aleator, Tomek Links, NearMiss) și le compari.
* Evaluezi clasificatorii pe date dezechilibrate folosind metricile potrivite: precision, recall, F1, ROC-AUC și PR-AUC.
* Implementezi SMOTE de la zero folosind NumPy și înțelegi pasul de interpolare.

Să începem!


---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">SFATURI PENTRU EVALUAREA CU SUCCES A TEMEI:</h4>

* Toate celulele sunt blocate, cu excepția celor în care trebuie să trimiți soluțiile sau când este menționat explicit că poți interacționa cu ele.

* În fiecare celulă de exercițiu, caută comentariile `### ÎNCEPUT DE COD AICI ###` și `### SFÂRȘIT DE COD AICI ###`. Acestea îți arată unde să scrii codul soluției. **Nu adăuga și nu modifica niciun cod în afara acestor comentarii**.

* Poți adăuga celule noi pentru experimente, dar acestea vor fi ignorate de evaluator, așa că nu te baza pe celulele create de tine pentru codul soluției; folosește spațiile oferite în notebook.

* Evită folosirea variabilelor globale, cu excepția situațiilor absolut necesare. Evaluatorul îți testează codul într-un mediu izolat fără să ruleze toate celulele de la început. Drept urmare, variabilele globale pot să nu fie disponibile în timpul evaluării. Variabilele globale care trebuie folosite vor fi definite cu MAJUSCULE.

* Pentru a trimite notebook-ul la evaluare, salvează-l mai întâi apăsând pe iconița 💾 din stânga sus a paginii, apoi apasă pe butonul `Submit assignment` din dreapta sus.
---


## Cuprins
- [Importuri](#imports)
- [1 - Problema Dezechilibrului de Clasă](#1)
    - [1.1 - De Ce Acuratețea Minte](#1-1)
    - [1.2 - Construirea unui Set de Date Dezechilibrat](#1-2)
- [2 - Diagnosticarea Dezechilibrului](#2)
    - **[Exercițiul 1 - diagnose_imbalance](#ex-1)**
- [3 - Abordarea cu Ponderi de Clasă](#3)
    - [3.1 - Cum Funcționează Ponderile de Clasă](#3-1)
    - **[Exercițiul 2 - train_with_class_weights](#ex-2)**
- [4 - Oversampling cu SMOTE](#4)
    - [4.1 - Algoritmul SMOTE](#4-1)
    - **[Exercițiul 3 - apply_smote](#ex-3)**
- [5 - Strategii de Undersampling](#5)
    - [5.1 - Trei Metode de Undersampling](#5-1)
    - **[Exercițiul 4 - apply_undersampling](#ex-4)**
- [6 - Evaluarea pe Date Dezechilibrate](#6)
    - [6.1 - Metricile Care Spun Adevărul](#6-1)
    - **[Exercițiul 5 - evaluate_imbalanced_classifier](#ex-5)**
- [7 - SMOTE de la Zero](#7)
    - [7.1 - Pasul de Interpolare](#7-1)
    - **[Exercițiul 6 - smote_from_scratch](#ex-6)**


<a name='imports'></a>
## Importuri


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import unittests

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    classification_report, confusion_matrix,
    RocCurveDisplay, PrecisionRecallDisplay
)

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler, TomekLinks, NearMiss

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

<a name='1'></a>
## 1 - Problema Dezechilibrului de Clasă

Înainte să trecem la soluții, să înțelegem ce efect au de fapt datele dezechilibrate asupra modelului tău.


<a name='1-1'></a>
### 1.1 - De Ce Acuratețea Minte

Acuratețea numără predicțiile corecte din totalul predicțiilor:

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

Când o clasă conține 99% din eșantioane, dacă prezici acea clasă pentru tot obții 99% acuratețe. Modelul este inutil, dar numărul arată excelent.

Soluția nu este să antrenezi mai agresiv — ci să folosești metrici mai bune și să regândești modul în care este antrenat modelul.

Cele două metrici care spun adevărul pentru clasa minoritară sunt:

$$\text{Precision} = \frac{TP}{TP + FP} \qquad \text{Recall} = \frac{TP}{TP + FN}$$

Iar scorul F1 le combină într-un singur număr folosind media armonică:

$$F1 = 2 \cdot \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$


<a name='1-2'></a>
### 1.2 - Construirea unui Set de Date Dezechilibrat

Vom crea un set de date sintetic dezechilibrat cu 10% clasă minoritară, similar cu ceea ce ai putea întâlni în detectarea fraudei.


In [ ]:
# Creează un set de date binar dezechilibrat pentru clasificare
X, y = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=5,
    n_redundant=2,
    weights=[0.90, 0.10],   # 90% majoritară, 10% minoritară
    flip_y=0.01,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f"Eșantioane de antrenare: {X_train.shape[0]}")
print(f"Eșantioane de test:      {X_test.shape[0]}")
print(f"Distribuția claselor în antrenare: {dict(zip(*np.unique(y_train, return_counts=True)))}")


Să vedem ce obține un model naiv, fără nicio tehnică de tratare a dezechilibrului:


In [ ]:
naive_model = LogisticRegression(random_state=42, max_iter=1000)
naive_model.fit(X_train, y_train)
y_pred_naive = naive_model.predict(X_test)

print("Model naiv (fără tratarea dezechilibrului):")
print(f"  Accuracy: {accuracy_score(y_test, y_pred_naive):.3f}")
print(f"  F1 Score (clasa minoritară): {f1_score(y_test, y_pred_naive):.3f}")
print()
print(classification_report(y_test, y_pred_naive, target_names=['Majoritară', 'Minoritară']))


<a name='2'></a>
## 2 - Diagnosticarea Dezechilibrului

Primul pas este întotdeauna să înțelegi distribuția claselor. Ai nevoie de cifre, nu doar de o impresie.


<a name='ex-1'></a>
#### **Exercițiul 1 - `diagnose_imbalance`**

**Sarcina ta:**

Implementează funcția `diagnose_imbalance`, care calculează statisticile distribuției claselor pentru un vector țintă.

Trebuie să implementezi:

* **Numărarea eșantioanelor per clasă**:
    * Folosește `np.unique(y, return_counts=True)` pentru a obține clasele unice și numărul lor de apariții.

* **Identificarea clasei majoritare și a celei minoritare**:
    * Clasa majoritară are cel mai mare număr de exemple: `classes[np.argmax(counts)]`.
    * Clasa minoritară are cel mai mic număr de exemple: `classes[np.argmin(counts)]`.

* **Calculul raportului de dezechilibru**:
    * `imbalance_ratio = counts.max() / counts.min()`

* **Returnarea unui dicționar** cu cheile: `class_counts`, `imbalance_ratio`, `majority_class`, `minority_class`.

<details>
  <summary><b><font color="green">Indicii suplimentare de cod (apasă pentru a extinde dacă te-ai blocat)</font></b></summary>

Dacă ai nevoie de puțin ajutor:

**Obținerea numărătorilor:**
* `classes, counts = np.unique(y, return_counts=True)`

**Construirea dicționarului:**
* `"class_counts": dict(zip(classes, counts))`

**Raportul de dezechilibru:**
* `counts.max() / counts.min()`

</details>


In [ ]:
# GRADED FUNCTION: diagnose_imbalance
def diagnose_imbalance(y):
    """
    Calculează statisticile distribuției claselor.

    Args:
        y: array cu etichete de clasă

    Returns:
        dict cu cheile:
            class_counts    -- dict care mapează eticheta clasei la numărul de eșantioane
            imbalance_ratio -- majority_count / minority_count
            majority_class  -- eticheta clasei cu cele mai multe eșantioane
            minority_class  -- eticheta clasei cu cele mai puține eșantioane
    """
    ### ÎNCEPUT DE COD AICI ###

    # Obține clasele unice și numărul lor de apariții

    # Găsește etichetele claselor majoritară și minoritară

    # Calculează raportul de dezechilibru

    ### SFÂRȘIT DE COD AICI ###

    return {
        "class_counts":    None,
        "imbalance_ratio": None,
        "majority_class":  None,
        "minority_class":  None,
    }


In [ ]:
# Testează implementarea ta
info = diagnose_imbalance(y_train)

print("Numărul de exemple per clasă: ", info["class_counts"])
print("Clasa majoritară:             ", info["majority_class"])
print("Clasa minoritară:             ", info["minority_class"])
print(f"Raport de dezechilibru:       {info['imbalance_ratio']:.1f}:1")

# Vizualizare
counts = info["class_counts"]
plt.figure(figsize=(6, 4))
plt.bar([str(k) for k in counts.keys()], list(counts.values()), color=['steelblue', 'coral'])
plt.title("Distribuția Claselor în Setul de Antrenare")
plt.xlabel("Clasă")
plt.ylabel("Număr de Eșantioane")
for i, (k, v) in enumerate(counts.items()):
    plt.text(i, v + 5, str(v), ha='center')
plt.show()


##### **Rezultatul Așteptat**
```
Numărul de exemple per clasă:  {0: 672, 1: 78}
Clasa majoritară:              0
Clasa minoritară:              1
Raport de dezechilibru:        8.6:1
```


In [ ]:
unittests.exercise_1(diagnose_imbalance)

<a name='3'></a>
## 3 - Abordarea cu Ponderi de Clasă

Cea mai simplă soluție pentru dezechilibru. O singură modificare de parametru care îi spune modelului: *acordă mai multă atenție greșelilor pe clasa minoritară*.


<a name='3-1'></a>
### 3.1 - Cum Funcționează Ponderile de Clasă

Setarea `class_weight='balanced'` într-un clasificator sklearn calculează automat ponderile:

$$w_c = \frac{n\_samples}{n\_classes \times n\_samples_c}$$

Aceste ponderi sunt aplicate funcției de pierdere. Un eșantion din clasa minoritară, cu pondere 5, contribuie de 5 ori mai mult la pierderea totală decât un eșantion din clasa majoritară, cu pondere 0.5. Modelul nu își mai poate permite să ignore clasa minoritară.


<a name='ex-2'></a>
#### **Exercițiul 2 - `train_with_class_weights`**

**Sarcina ta:**

Implementează funcția `train_with_class_weights`, care antrenează un model `LogisticRegression` de două ori: o dată fără ponderi de clasă și o dată cu `class_weight='balanced'`. Returnează ambele scoruri F1 și ambele modele.

Trebuie să implementezi:

* **Antrenarea fără ponderi de clasă**:
    * `LogisticRegression(class_weight=None, max_iter=1000, random_state=42)`
    * Antrenează pe `X_train`, `y_train`. Calculează F1 pe `X_test`, `y_test`.

* **Antrenarea cu ponderi de clasă echilibrate**:
    * `LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)`
    * Aceiași pași ca mai sus.

* **Returnarea unui dicționar** cu cheile: `f1_none`, `f1_balanced`, `model_none`, `model_balanced`.

<details>
  <summary><b><font color="green">Indicii suplimentare de cod (apasă pentru a extinde dacă te-ai blocat)</font></b></summary>

Dacă ai nevoie de puțin ajutor:

**Scor F1:**
* `f1_score(y_test, model.predict(X_test))`

**Returnarea dicționarului:**
* `return {"f1_none": ..., "f1_balanced": ..., "model_none": ..., "model_balanced": ...}`

</details>


In [ ]:
# GRADED FUNCTION: train_with_class_weights
def train_with_class_weights(X_train, X_test, y_train, y_test):
    """
    Antrenează LogisticRegression cu și fără class_weight='balanced'.

    Args:
        X_train, X_test: array-uri de caracteristici
        y_train, y_test: array-uri de etichete

    Returns:
        dict cu cheile:
            f1_none        -- scorul F1 (clasa minoritară) fără ponderi de clasă
            f1_balanced    -- scorul F1 (clasa minoritară) cu class_weight='balanced'
            model_none     -- modelul antrenat fără ponderi de clasă
            model_balanced -- modelul antrenat cu class_weight='balanced'
    """
    ### ÎNCEPUT DE COD AICI ###

    # Antrenează fără ponderi de clasă și calculează F1

    # Antrenează cu class_weight='balanced' și calculează F1

    ### SFÂRȘIT DE COD AICI ###

    return {
        "f1_none":       None,
        "f1_balanced":   None,
        "model_none":    None,
        "model_balanced": None,
    }


In [ ]:
# Testează implementarea ta
results_cw = train_with_class_weights(X_train, X_test, y_train, y_test)

print("Scor F1 (clasa minoritară):")
print(f"  Fără ponderi de clasă:      {results_cw['f1_none']:.3f}")
print(f"  Cu class_weight='balanced': {results_cw['f1_balanced']:.3f}")

labels = ['Fără ponderi', 'Balanced']
scores = [results_cw['f1_none'], results_cw['f1_balanced']]
plt.figure(figsize=(6, 4))
plt.bar(labels, scores, color=['steelblue', 'coral'])
plt.ylim(0, 1)
plt.title("Scor F1: Fără Ponderi vs. Ponderi de Clasă Echilibrate")
plt.ylabel("Scor F1 (clasa minoritară)")
for i, s in enumerate(scores):
    plt.text(i, s + 0.02, f"{s:.3f}", ha='center')
plt.show()


##### **Rezultatul Așteptat**
```
Scor F1 (clasa minoritară):
  Fără ponderi de clasă:      0.333
  Cu class_weight='balanced': 0.423
```


In [ ]:
unittests.exercise_2(train_with_class_weights)

<a name='4'></a>
## 4 - Oversampling cu SMOTE

Ponderile de clasă ajustează funcția de pierdere, dar nu schimbă datele. SMOTE merge mai departe și generează noi eșantioane sintetice din clasa minoritară.


<a name='4-1'></a>
### 4.1 - Algoritmul SMOTE

Pentru fiecare eșantion minoritar $x_i$, SMOTE găsește cei $k$ cei mai apropiați vecini minoritari, alege unul ($x_{nn}$), apoi creează un punct nou:

$$x_{new} = x_i + \lambda \cdot (x_{nn} - x_i), \quad \lambda \sim \text{Uniform}(0, 1)$$

Noul punct cade undeva pe segmentul dintre două eșantioane minoritare reale. Este plauzibil, dar nu este o dublură.

**Regula cheie**: aplică SMOTE doar pe setul de antrenare. Setul de test trebuie să rămână neatins.


<a name='ex-3'></a>
#### **Exercițiul 3 - `apply_smote`**

**Sarcina ta:**

Implementează funcția `apply_smote`, care aplică oversampling SMOTE pe setul de antrenare și returnează datele resamplate.

Trebuie să implementezi:

* **Crearea unui obiect SMOTE**: `SMOTE(k_neighbors=k_neighbors, random_state=random_state)`
* **Resamplarea**: `smote.fit_resample(X_train, y_train)`
* **Returnarea** lui `X` și `y` resamplate.

<details>
  <summary><b><font color="green">Indicii suplimentare de cod (apasă pentru a extinde dacă te-ai blocat)</font></b></summary>

Dacă ai nevoie de puțin ajutor:

```python
from imblearn.over_sampling import SMOTE   # deja importat
smote = SMOTE(k_neighbors=k_neighbors, random_state=random_state)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)
```

</details>


In [ ]:
# GRADED FUNCTION: apply_smote
def apply_smote(X_train, y_train, k_neighbors=5, random_state=42):
    """
    Aplică oversampling SMOTE pe datele de antrenare.

    Args:
        X_train:      array de caracteristici pentru antrenare
        y_train:      array de etichete pentru antrenare
        k_neighbors:  numărul de vecini cei mai apropiați pentru SMOTE
        random_state: seed aleator

    Returns:
        X_resampled, y_resampled: array-uri resamplate
    """
    ### ÎNCEPUT DE COD AICI ###

    # Creează obiectul SMOTE

    # Aplică fit și resample

    ### SFÂRȘIT DE COD AICI ###

    return X_resampled, y_resampled


In [ ]:
# Testează implementarea ta
X_smote, y_smote = apply_smote(X_train, y_train, k_neighbors=5)

print("Înainte de SMOTE:", {int(k): int(v) for k, v in zip(*np.unique(y_train, return_counts=True))})
print("După SMOTE:      ", {int(k): int(v) for k, v in zip(*np.unique(y_smote, return_counts=True))})

# Antrenează un model pe datele resamplate cu SMOTE
model_smote = LogisticRegression(random_state=42, max_iter=1000)
model_smote.fit(X_smote, y_smote)
y_pred_smote = model_smote.predict(X_test)

print(f"\nF1 (clasa minoritară) după SMOTE: {f1_score(y_test, y_pred_smote):.3f}")


##### **Rezultatul Așteptat**
```
Înainte de SMOTE: {0: 672, 1: 78}
După SMOTE:       {0: 672, 1: 672}

F1 (clasa minoritară) după SMOTE: 0.455
```


In [ ]:
unittests.exercise_3(apply_smote)

### 4.2 - Ajustarea lui k_neighbors

Să vedem cum influențează valorile diferite ale lui `k_neighbors` scorul F1:


In [ ]:
k_values = [1, 3, 5, 7, 10]
f1_scores = []

for k in k_values:
    X_res, y_res = apply_smote(X_train, y_train, k_neighbors=k)
    clf = LogisticRegression(random_state=42, max_iter=1000)
    clf.fit(X_res, y_res)
    f1_scores.append(f1_score(y_test, clf.predict(X_test)))

plt.figure(figsize=(7, 4))
plt.plot(k_values, f1_scores, marker='o', color='coral')
plt.title("SMOTE: Efectul lui k_neighbors asupra scorului F1")
plt.xlabel("k_neighbors")
plt.ylabel("Scor F1 (clasa minoritară)")
plt.xticks(k_values)
plt.show()


<a name='5'></a>
## 5 - Strategii de Undersampling

În loc să creeze mai multe eșantioane minoritare, undersampling-ul elimină eșantioane majoritare. Trei strategii, fiecare cu o logică diferită.


<a name='5-1'></a>
### 5.1 - Trei Metode de Undersampling

| Metodă | Ce elimină |
|:-------|:-----------|
| **Random** | Eșantioane majoritare alese aleator |
| **Tomek Links** | Eșantioane majoritare care formează legături Tomek cu eșantioane minoritare (perechi de frontieră) |
| **NearMiss-1** | Păstrează eșantioanele majoritare cele mai apropiate de clasa minoritară |

Toate trei returnează un set de antrenare resamplat. Setul de test nu este atins niciodată.


<a name='ex-4'></a>
#### **Exercițiul 4 - `apply_undersampling`**

**Sarcina ta:**

Implementează funcția `apply_undersampling`, care aplică una dintre cele trei strategii de undersampling în funcție de argumentul `strategy`.

Trebuie să implementezi:

* **Dacă `strategy == 'random'`**: folosește `RandomUnderSampler(random_state=random_state)`
* **Dacă `strategy == 'tomek'`**: folosește `TomekLinks()`
* **Dacă `strategy == 'nearmiss'`**: folosește `NearMiss(version=1)`
* **Ridică `ValueError`** pentru orice alt șir de strategie.
* **Returnează** `X_resampled, y_resampled` din `sampler.fit_resample(X_train, y_train)`.

<details>
  <summary><b><font color="green">Indicii suplimentare de cod (apasă pentru a extinde dacă te-ai blocat)</font></b></summary>

Dacă ai nevoie de puțin ajutor:

```python
if strategy == 'random':
    sampler = RandomUnderSampler(random_state=random_state)
elif strategy == 'tomek':
    sampler = TomekLinks()
elif strategy == 'nearmiss':
    sampler = NearMiss(version=1)
else:
    raise ValueError(f"Unknown strategy: {strategy}")
```

</details>


In [ ]:
# GRADED FUNCTION: apply_undersampling
def apply_undersampling(X_train, y_train, strategy='random', random_state=42):
    """
    Aplică o strategie de undersampling pe datele de antrenare.

    Args:
        X_train:      array de caracteristici pentru antrenare
        y_train:      array de etichete pentru antrenare
        strategy:     una dintre 'random', 'tomek', 'nearmiss'
        random_state: seed aleator (folosit de Random și NearMiss)

    Returns:
        X_resampled, y_resampled: array-uri resamplate
    """
    ### ÎNCEPUT DE COD AICI ###

    # Selectează metoda de undersampling în funcție de strategie

    # Aplică fit și resample

    ### SFÂRȘIT DE COD AICI ###

    return X_resampled, y_resampled


In [ ]:
# Testează implementarea ta cu toate cele trei strategii
for strat in ['random', 'tomek', 'nearmiss']:
    X_res, y_res = apply_undersampling(X_train, y_train, strategy=strat)
    clf = LogisticRegression(random_state=42, max_iter=1000)
    clf.fit(X_res, y_res)
    f1 = f1_score(y_test, clf.predict(X_test))
    dist = {int(k): int(v) for k, v in zip(*np.unique(y_res, return_counts=True))}
    print(f"{strat:10s} | distribuție: {dist} | F1: {f1:.3f}")


##### **Rezultatul Așteptat**
```
random     | distribuție: {0: 78, 1: 78}   | F1: 0.415
tomek      | distribuție: {0: 667, 1: 78}  | F1: 0.278
nearmiss   | distribuție: {0: 78, 1: 78}   | F1: 0.379
```


In [ ]:
unittests.exercise_4(apply_undersampling)

<a name='6'></a>
## 6 - Evaluarea pe Date Dezechilibrate

Acum avem mai multe modele antrenate cu strategii diferite. Să construim o funcție de evaluare corectă care calculează metricile ce contează cu adevărat.


<a name='6-1'></a>
### 6.1 - Metricile Care Spun Adevărul

Pentru clasificarea dezechilibrată ai nevoie de:

| Metrică | Formulă | Ce îți spune |
|:--------|:--------|:-------------|
| **Precision** | $TP / (TP + FP)$ | Din toate pozitivele prezise, câte sunt reale? |
| **Recall** | $TP / (TP + FN)$ | Din toate pozitivele reale, pe câte le-am găsit? |
| **F1** | media armonică dintre Precision și Recall | Un singur număr care echilibrează ambele |
| **ROC-AUC** | Aria de sub curba TPR vs FPR | Capacitatea generală de ordonare pe toate pragurile |
| **PR-AUC** | Aria de sub curba Precision-Recall | Cel mai bun rezumat pentru date dezechilibrate |


<a name='ex-5'></a>
#### **Exercițiul 5 - `evaluate_imbalanced_classifier`**

**Sarcina ta:**

Implementează funcția `evaluate_imbalanced_classifier`, care calculează toate cele cinci metrici și le returnează ca dicționar.

Trebuie să implementezi:

* **Precision**: `precision_score(y_true, y_pred)`
* **Recall**: `recall_score(y_true, y_pred)`
* **F1**: `f1_score(y_true, y_pred)`
* **ROC-AUC**: `roc_auc_score(y_true, y_prob)`
* **PR-AUC**: `average_precision_score(y_true, y_prob)`

Returnează un dicționar cu cheile: `precision`, `recall`, `f1`, `roc_auc`, `pr_auc`.

<details>
  <summary><b><font color="green">Indicii suplimentare de cod (apasă pentru a extinde dacă te-ai blocat)</font></b></summary>

Dacă ai nevoie de puțin ajutor:

* `y_pred` provine din `model.predict(X_test)` — etichete hard de clasă.
* `y_prob` provine din `model.predict_proba(X_test)[:, 1]` — probabilitatea clasei pozitive.
* Precision, recall și F1 folosesc `y_pred`. ROC-AUC și PR-AUC folosesc `y_prob`.

</details>


In [ ]:
# GRADED FUNCTION: evaluate_imbalanced_classifier
def evaluate_imbalanced_classifier(y_true, y_pred, y_prob):
    """
    Calculează metricile de evaluare pentru un clasificator binar dezechilibrat.

    Args:
        y_true: etichetele binare reale
        y_pred: etichetele binare prezise (din model.predict)
        y_prob: probabilitățile prezise pentru clasa pozitivă
                (din model.predict_proba[:, 1])

    Returns:
        dict cu cheile: precision, recall, f1, roc_auc, pr_auc
    """
    ### ÎNCEPUT DE COD AICI ###

    # Calculează toate cele cinci metrici

    ### SFÂRȘIT DE COD AICI ###

    return {
        "precision": None,
        "recall":    None,
        "f1":        None,
        "roc_auc":   None,
        "pr_auc":    None,
    }


In [ ]:
# Testează implementarea ta comparând strategiile
strategy_results = {}

# Naiv (fără tratare)
y_pred_n = naive_model.predict(X_test)
y_prob_n = naive_model.predict_proba(X_test)[:, 1]
strategy_results['naive'] = evaluate_imbalanced_classifier(y_test, y_pred_n, y_prob_n)

# Ponderi de clasă
y_pred_cw = results_cw['model_balanced'].predict(X_test)
y_prob_cw = results_cw['model_balanced'].predict_proba(X_test)[:, 1]
strategy_results['class_weights'] = evaluate_imbalanced_classifier(y_test, y_pred_cw, y_prob_cw)

# SMOTE
y_pred_sm = model_smote.predict(X_test)
y_prob_sm = model_smote.predict_proba(X_test)[:, 1]
strategy_results['smote'] = evaluate_imbalanced_classifier(y_test, y_pred_sm, y_prob_sm)

# Afișează tabelul comparativ
metrics_df = pd.DataFrame(strategy_results).T
print(metrics_df.round(3).to_string())


##### **Rezultatul Așteptat**
```
              precision  recall     f1  roc_auc  pr_auc
naive              0.600   0.231  0.333    0.842   0.444
class_weights      0.282   0.846  0.423    0.849   0.316
smote              0.307   0.885  0.455    0.850   0.305
```


In [ ]:
# Reprezintă curbele PR pentru fiecare strategie
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models_to_plot = {
    'Naiv':          (naive_model, y_prob_n),
    'Ponderi de clasă': (results_cw['model_balanced'], y_prob_cw),
    'SMOTE':         (model_smote, y_prob_sm),
}

for name, (_, y_prob) in models_to_plot.items():
    PrecisionRecallDisplay.from_predictions(y_test, y_prob, name=name, ax=axes[0])
    RocCurveDisplay.from_predictions(y_test, y_prob, name=name, ax=axes[1])

axes[0].set_title("Curbe Precision-Recall")
axes[1].set_title("Curbe ROC")
plt.tight_layout()
plt.show()


In [ ]:
unittests.exercise_5(evaluate_imbalanced_classifier)

<a name='7'></a>
## 7 - SMOTE de la Zero

Acum să construim SMOTE fără să folosim nicio bibliotecă. Asta îți va arăta exact ce face algoritmul, pas cu pas.


<a name='7-1'></a>
### 7.1 - Pasul de Interpolare

Nucleul algoritmului SMOTE este această linie:

$$x_{new} = x_i + \lambda \cdot (x_{nn} - x_i), \quad \lambda \in [0, 1]$$

Când $\lambda = 0$, obții chiar $x_i$. Când $\lambda = 1$, obții $x_{nn}$. Pentru orice valoare intermediară, obții un punct pe segmentul dintre ele — un nou eșantion minoritar sintetic.

O subtilitate: când antrenezi k-NN pe eșantioanele minoritare și apoi interoghezi unul dintre aceleași eșantioane, cel mai apropiat vecin la distanța 0 este chiar el însuși. Trebuie să antrenezi cu `k + 1` vecini și să ignori primul rezultat.


<a name='ex-6'></a>
#### **Exercițiul 6 - `smote_from_scratch`**

**Sarcina ta:**

Implementează SMOTE de la zero folosind NumPy și `sklearn.neighbors.NearestNeighbors`.

Trebuie să implementezi:

1. **Separarea claselor**: folosește `np.unique` pentru a găsi clasele majoritară și minoritară. Împarte `X` în `X_min` și `X_maj`.

2. **Calculul numărului de eșantioane sintetice necesare**: `n_synthetic = len(X_maj) - len(X_min)`

3. **Antrenarea k-NN pe eșantioanele minoritare**: `NearestNeighbors(n_neighbors=k_neighbors + 1)` — acel +1 apare deoarece eșantionul însuși va fi propriul vecin cel mai apropiat.

4. **Generarea eșantioanelor sintetice** (buclă de `n_synthetic` ori):
    * Alege un indice minoritar aleator `idx`.
    * Obține `x_i = X_min[idx]`.
    * Găsește vecinii săi: `nn.kneighbors([x_i], return_distance=False)[0]`.
    * Ignoră indexul 0 (self), alege un vecin aleator `x_nn`.
    * Extrage `lam = rng.uniform(0, 1)`.
    * Creează `x_new = x_i + lam * (x_nn - x_i)`.

5. **Stack și return**: `np.vstack([X, X_synthetic])` și `np.concatenate([y, y_synthetic])`.

<details>
  <summary><b><font color="green">Indicii suplimentare de cod (apasă pentru a extinde dacă te-ai blocat)</font></b></summary>

Dacă ai nevoie de puțin ajutor:

```python
rng = np.random.default_rng(random_state)
classes, counts = np.unique(y, return_counts=True)
majority_class = classes[np.argmax(counts)]
minority_class = classes[np.argmin(counts)]
X_min = X[y == minority_class]
X_maj = X[y == majority_class]
n_synthetic = len(X_maj) - len(X_min)

nn_model = NearestNeighbors(n_neighbors=k_neighbors + 1)
nn_model.fit(X_min)
```

</details>


In [ ]:
# GRADED FUNCTION: smote_from_scratch
def smote_from_scratch(X, y, k_neighbors=5, random_state=42):
    """
    Implementează SMOTE de la zero folosind NumPy.

    Args:
        X:            array de caracteristici
        y:            array de etichete (binar)
        k_neighbors:  numărul de vecini minoritari apropiați luați în calcul
        random_state: seed aleator

    Returns:
        X_resampled, y_resampled: array-uri la care s-au adăugat eșantioane sintetice minoritare
    """
    rng = np.random.default_rng(random_state)

    ### ÎNCEPUT DE COD AICI ###

    # 1. Separă eșantioanele majoritare și minoritare

    # 2. Calculează numărul de eșantioane sintetice necesare

    # 3. Antrenează k-NN pe eșantioanele minoritare (k_neighbors + 1)

    # 4. Generează eșantioane sintetice
    synthetic_samples = []
    for _ in range(n_synthetic):
        # Alege un eșantion minoritar aleator

        # Obține vecinii lui cei mai apropiați (ignoră indexul 0 — acela este chiar eșantionul)

        # Alege un vecin aleator

        # Interpolează

        pass  # elimină această linie după ce completezi bucla

    # 5. Concatenează eșantioanele sintetice cu datele originale

    ### SFÂRȘIT DE COD AICI ###

    return X_resampled, y_resampled


In [ ]:
# Testează implementarea ta
X_scratch, y_scratch = smote_from_scratch(X_train, y_train, k_neighbors=5)

print("Înainte (SMOTE de la zero):", {int(k): int(v) for k, v in zip(*np.unique(y_train, return_counts=True))})
print("După     (SMOTE de la zero):", {int(k): int(v) for k, v in zip(*np.unique(y_scratch, return_counts=True))})

# Antrenează un model pe datele SMOTE construite de la zero
model_scratch = LogisticRegression(random_state=42, max_iter=1000)
model_scratch.fit(X_scratch, y_scratch)
y_pred_scratch = model_scratch.predict(X_test)

print(f"\nF1 (clasa minoritară) cu SMOTE de la zero: {f1_score(y_test, y_pred_scratch):.3f}")
print("(Ar trebui să fie apropiat de rezultatul SMOTE din imblearn)")


##### **Rezultatul Așteptat**
```
Înainte (SMOTE de la zero): {0: 672, 1: 78}
După     (SMOTE de la zero): {0: 672, 1: 672}

F1 (clasa minoritară) cu SMOTE de la zero: 0.465
(Ar trebui să fie apropiat de rezultatul SMOTE din imblearn)
```


In [ ]:
unittests.exercise_6(smote_from_scratch)

## Rezumat

Felicitări pentru finalizarea temei! Iată ce ai parcurs:

| Exercițiu | Ce ai construit | Ideea-cheie |
|:----------|:----------------|:------------|
| 1 | `diagnose_imbalance` | Măsoară întotdeauna mai întâi raportul de dezechilibru |
| 2 | `train_with_class_weights` | O singură modificare de parametru poate îmbunătăți semnificativ scorul F1 |
| 3 | `apply_smote` | SMOTE creează eșantioane sintetice, nu duplicate |
| 4 | `apply_undersampling` | Trei strategii cu compromisuri diferite |
| 5 | `evaluate_imbalanced_classifier` | Acuratețea minte — folosește F1 și PR-AUC |
| 6 | `smote_from_scratch` | SMOTE este, în esență, interpolare între vecini minoritari |

**Regulile de aur pentru date dezechilibrate:**
1. Nu folosi niciodată acuratețea ca metrică principală.
2. Începe simplu — încearcă mai întâi `class_weight='balanced'`.
3. Aplică SMOTE doar pe setul de antrenare.
4. Folosește PR-AUC sau scorul F1 pentru a compara strategiile.


# Tema de Programare: Tratarea Datelor Dezechilibrate

Bine ai venit la tema despre tratarea datelor dezechilibrate.

Dezechilibrul de clasă este una dintre cele mai comune probleme cu care te vei confrunta în machine learning-ul din lumea reală. Detectarea fraudei, diagnosticul medical, detectarea defectelor — în toate aceste cazuri, ceea ce îți pasă cel mai mult să prezici este elementul rar. Un model care îl ignoră poate arăta excelent după acuratețe și totuși să fie complet inutil.

În această temă vei parcurge întregul pipeline pentru gestionarea datelor dezechilibrate: diagnosticarea problemei, corectarea ei și evaluarea corectă a modelului. Vei folosi tehnici reale de resamplare — ponderi de clasă, SMOTE, Tomek Links și NearMiss — și vei învăța care metrici de evaluare spun cu adevărat povestea corectă.

Tema se încheie cu o implementare de la zero a algoritmului SMOTE, astfel încât să înțelegi exact ce face biblioteca în culise.

**Ce vei face în această temă**

* Diagnostichezi dezechilibrul de clasă și cuantifici cât de sever este.
* Aplici ponderi de clasă pentru ca clasificatorii să acorde atenție clasei minoritare.
* Folosești SMOTE pentru a genera eșantioane sintetice minoritare și îi ajustezi parametrii.
* Aplici strategii de undersampling (aleator, Tomek Links, NearMiss) și le compari.
* Evaluezi clasificatorii pe date dezechilibrate folosind metricile potrivite: precision, recall, F1, ROC-AUC și PR-AUC.
* Implementezi SMOTE de la zero folosind NumPy și înțelegi pasul de interpolare.

Să începem!
